# Cell 1 — Imports

In [1]:
import pickle
import torch
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import seaborn as sns
import matplotlib.pyplot as plt

# Cell 2 — Load ML Dataset Split

In [2]:
with open("../data/processed/ml/ml_train_test_split.pkl", "rb") as f:
    data = pickle.load(f)

X_test_ml = data["X_test"]
y_test_ml = data["y_test"]

# Cell 3 — Load ML Models

In [3]:
import joblib

dt = joblib.load("../models/ml/decision_tree.pkl")
rf = joblib.load("../models/ml/random_forest.pkl")
svm = joblib.load("../models/ml/svm.pkl")
knn = joblib.load("../models/ml/knn.pkl")

# Cell 4 — ML Predictions

In [4]:
ml_predictions = {}

ml_predictions["Decision Tree"] = dt.predict(X_test_ml)
ml_predictions["Random Forest"] = rf.predict(X_test_ml)
ml_predictions["SVM"] = svm.predict(X_test_ml)
ml_predictions["KNN"] = knn.predict(X_test_ml)

c:\Users\kaush\Desktop\LLM Threat Detection on IIOT\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LinearSVC was fitted without feature names
  warnings.warn(
c:\Users\kaush\Desktop\LLM Threat Detection on IIOT\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


In [6]:
pip install pyarrow

   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.6 MB ? eta -:--:--
    --------------------------------------- 0.5/27.6 MB 3.4 MB/s eta 0:00:09
    --------------------------------------- 0.5/27.6 MB 3.4 MB/s eta 0:00:09
    --------------------------------------- 0.5/27.6 MB 3.4 MB/s eta 0:00:09
    --------------------------------------- 0.5/27.6 MB 3.4 MB/s eta 0:00:09
    --------------------------------------- 0.5/27.6 MB 3.4 MB/s eta 0:00:09
   - -------------------------------------- 0.8/27.6 MB 419.4 kB/s eta 0:01:04
   - -------------------------------------- 1.3/27.6 MB 745.8 kB/s eta 0:00:36
   -- ------------------------------------- 1.6/27.6 MB 799.2 kB/s eta 0:00:33
   -- ------------------------------------- 1.8/27.6 MB 883.1 kB/s eta 0:00:30
   --- ------------------------------------ 2.1/27.6 MB 939.6 kB/s eta 0:00:28
   --- ---

# Cell 5 — Load DL Dataset

In [7]:
with open("../data/processed/dl/dl_train_test_split.pkl", "rb") as f:
    data_dl = pickle.load(f)

X_test_dl = data_dl["X_test"]
y_test_dl = data_dl["y_test"]

ImportError: pyarrow>=13.0.0 is required for PyArrow backed StringArray.

# Cell 6 — Load DL Models

In [ ]:
from dnn_model import DNNModel
from lstm_model import LSTMModel
from cnn_model import CNNModel

# Cell 7 — Load DNN

In [ ]:
dnn = torch.load("../models/dl/dnn_model.pth")
dnn.eval()

# Cell 8 — Load LSTM

In [ ]:
lstm = torch.load("../models/dl/lstm_model.pth")
lstm.eval()

# Cell 9 — Load CNN

In [ ]:
cnn = torch.load("../models/dl/cnn_model.pth")
cnn.eval()

# Cell 10 — DL Predictions

In [ ]:
def get_dl_predictions(model, X):

    with torch.no_grad():

        outputs = model(torch.tensor(X.values, dtype=torch.float32))
        _, preds = torch.max(outputs, 1)

    return preds.numpy()


dl_predictions = {}

dl_predictions["DNN"] = get_dl_predictions(dnn, X_test_dl)
dl_predictions["LSTM"] = get_dl_predictions(lstm, X_test_dl)
dl_predictions["CNN"] = get_dl_predictions(cnn, X_test_dl)

# Cell 11 — Load SecurityBERT

In [ ]:
from securitybert_model import SecurityBERT

model = SecurityBERT(vocab_size=30522)

model.load_state_dict(
    torch.load("../models/securitybert/best.pt")
)

model.eval()

# Cell 12 — SecurityBERT Prediction Function

In [ ]:
def predict_securitybert(model, dataloader):

    preds = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"]

            outputs = model(input_ids)

            _, predicted = torch.max(outputs, 1)

            preds.extend(predicted.numpy())

    return np.array(preds)

# Cell 13 — Metrics Function

In [ ]:
def compute_metrics(y_true, y_pred):

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted"),
        "recall": recall_score(y_true, y_pred, average="weighted"),
        "f1": f1_score(y_true, y_pred, average="weighted")
    }

# Cell 14 — Evaluate ML Models

In [ ]:
results = {}

for model_name, preds in ml_predictions.items():

    results[model_name] = compute_metrics(y_test_ml, preds)

# Cell 15 — Evaluate DL Models

In [ ]:
for model_name, preds in dl_predictions.items():

    results[model_name] = compute_metrics(y_test_dl, preds)

# Cell 16 — Evaluate SecurityBERT

In [ ]:
securitybert_preds = predict_securitybert(model, test_loader)

results["SecurityBERT"] = compute_metrics(y_test_dl, securitybert_preds)

# Cell 17 — Create Comparison Table

Example output:

| Model            | Accuracy      | Precision | Recall | F1    |
| ---------------- | ------------- | --------- | ------ | ----- |
| Decision Tree    | 0.98          | 0.98      | 0.98   | 0.98  |
| Random Forest    | 0.984         | 0.985     | 0.984  | 0.984 |
| SVM              | 0.96          | 0.96      | 0.96   | 0.96  |
| KNN              | 0.94          | 0.94      | 0.94   | 0.94  |
| DNN              | 0.95          | 0.95      | 0.95   | 0.95  |
| LSTM             | 0.96          | 0.96      | 0.96   | 0.96  |
| CNN              | 0.96          | 0.96      | 0.96   | 0.96  |
| **SecurityBERT** | **0.98–0.99** |           |        |       |

In [ ]:
results_df = pd.DataFrame(results).T

results_df

# Cell 18 — Accuracy Comparison Plot

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    x=results_df.index,
    y=results_df["accuracy"]
)

plt.xticks(rotation=45)

plt.title("Model Accuracy Comparison")

plt.show()

# Cell 19 — Save Results

In [ ]:
results_df.to_csv("../results/model_comparison.csv")

print("Saved comparison results.")